In [1]:
import sqlite3
connect = sqlite3.connect("../db/pl_data.db")
cursor = connect.cursor()

In [ ]:
# confirming total games
cursor.execute("SELECT season_year, COUNT(*) FROM matches GROUP BY season_year")
cursor.fetchall()

In [ ]:
# get teams for CSV matching
cursor.execute("SELECT team_id, name FROM teams")
cursor.fetchall()

In [ ]:
cursor.execute("SELECT season_year, COUNT(*) FROM matches GROUP BY season_year")
cursor.fetchall()

In [1]:
import pandas as pd
import sqlite3

connect = sqlite3.connect("../db/pl_data.db")
matches = pd.read_sql_query("SELECT * FROM matches", connect)

In [3]:
# home DataFrame
home = matches[["season_year", "home_team_id", "home_goals_fulltime", "away_goals_fulltime", "winner"]].copy()
home.columns = ["season_year", "team_id", "goals_for", "goals_against", "winner",]

# away DataFrame
away = matches[["season_year", "away_team_id", "away_goals_fulltime", "home_goals_fulltime", "winner"]].copy()
away.columns = ["season_year", "team_id", "goals_for", "goals_against", "winner",]

home_result_map = {
    "HOME_TEAM": "WIN",
    "AWAY_TEAM": "LOSS",
    "DRAW": "DRAW"
}

away_result_map = {
    "HOME_TEAM": "LOSS",
    "AWAY_TEAM": "WIN",
    "DRAW": "DRAW"
}

home["result"] = home["winner"].map(home_result_map)
away["result"] = away["winner"].map(away_result_map)

team_matches = pd.concat([home, away], ignore_index=True)

seasons = team_matches.groupby(["team_id", "season_year"]).agg(
    matches_played=("winner", "count"),
    total_goals_for=("goals_for", "sum"),
    total_goals_against=("goals_against", "sum"),
    wins=("result", lambda x: (x == "WIN").sum()),
    draws=("result", lambda x: (x == "DRAW").sum()),
    losses=("result", lambda x: (x == "LOSS").sum()),
)

seasons["points"] = seasons["wins"] * 3 + seasons["draws"]
seasons["goal_difference"] = seasons["total_goals_for"] - seasons["total_goals_against"] 
# seasons["position"] = seasons.groupby("season_year")["points"].rank(ascending=False)

seasons = seasons.sort_values(["season_year", "points", "goal_difference"], ascending=[True, False, False])
seasons["position"] = seasons.groupby("season_year").cumcount() + 1



seasons = seasons[["matches_played", "total_goals_for", "total_goals_against", "goal_difference", "wins", "draws", "losses", "points", "position"]]

In [6]:
seasons_flat = seasons.reset_index()
seasons_flat.head()

,team_id,season_year,matches_played,total_goals_for,total_goals_against,goal_difference,wins,draws,losses,points,position
0,65,2018,38,95,23,72,32,2,4,98,1
1,64,2018,38,89,22,67,30,7,1,97,2
2,61,2018,38,63,39,24,21,9,8,72,3
3,73,2018,38,67,39,28,23,2,13,71,4
4,57,2018,38,73,51,22,21,7,10,70,5


In [35]:
# rolling form

home = matches[["match_id", "season_year", "utc_date", "home_team_id", "home_goals_fulltime", "away_goals_fulltime", "winner"]].copy()
home.columns = ["match_id", "season_year", "date", "team_id", "goals_for", "goals_against", "winner"]

away = matches[["match_id", "season_year", "utc_date", "away_team_id", "away_goals_fulltime", "home_goals_fulltime", "winner"]].copy()
away.columns = ["match_id", "season_year", "date", "team_id", "goals_for", "goals_against", "winner"]

home_result_map = {
    "HOME_TEAM": "WIN",
    "AWAY_TEAM": "LOSS",
    "DRAW": "DRAW"
}

away_result_map = {
    "HOME_TEAM": "LOSS",
    "AWAY_TEAM": "WIN",
    "DRAW": "DRAW"
}

home["result"] = home["winner"].map(home_result_map)
away["result"] = away["winner"].map(away_result_map)

home["date"] = pd.to_datetime(home["date"], dayfirst=True)
away["date"] = pd.to_datetime(away["date"], dayfirst=True)

points_dict = {
    "WIN": 3,
    "DRAW": 1,
    "LOSS": 0
}

home["points"] = home["result"].map(points_dict)
away["points"] = away["result"].map(points_dict)

team_matches = pd.concat([home, away], ignore_index=True)
team_matches = team_matches.sort_values(["team_id", "date"], ascending=[True, True])

team_matches["rolling_gFor_5"] = team_matches.groupby("team_id")["goals_for"].shift(1).rolling(window=5).sum()
team_matches["rolling_gAgainst_5"] = team_matches.groupby("team_id")["goals_against"].shift(1).rolling(window=5).sum()
team_matches["rolling_points_5"] = team_matches.groupby("team_id")["points"].shift(1).rolling(window=5).sum()

# Split the rolling-feature table back into home-side and away-side pieces
# using the original matches table to know which team_id was home vs away

home_features = matches[["match_id", "home_team_id"]].merge(
    team_matches,
    left_on=["match_id", "home_team_id"],
    right_on=["match_id", "team_id"]
)

away_features = matches[["match_id", "away_team_id"]].merge(
    team_matches,
    left_on=["match_id", "away_team_id"],
    right_on=["match_id", "team_id"]
)

# merging back
match_features = home_features.merge(
    away_features,
    on="match_id",
    suffixes=("_home", "_away")
)

In [ ]:
team_matches[team_matches["team_id"] == 66].head(10)

In [38]:
match_features.head()

,match_id,home_team_id,season_year_home,date_home,team_id_home,goals_for_home,goals_against_home,winner_home,result_home,points_home,...,date_away,team_id_away,goals_for_away,goals_against_away,winner_away,result_away,points_away,rolling_gFor_5_away,rolling_gAgainst_5_away,rolling_points_5_away
0,20000,66,2018,2018-08-10,66,2,1,HOME_TEAM,WIN,3,...,2018-08-10,9007,1,2,HOME_TEAM,LOSS,0,NaN,NaN,NaN
1,20001,1044,2018,2018-08-11,1044,2,0,HOME_TEAM,WIN,3,...,2018-08-11,9003,0,2,HOME_TEAM,LOSS,0,NaN,NaN,NaN
2,20002,63,2018,2018-08-11,63,0,2,AWAY_TEAM,LOSS,0,...,2018-08-11,354,2,0,AWAY_TEAM,WIN,3,NaN,NaN,NaN
3,20003,9004,2018,2018-08-11,9004,0,3,AWAY_TEAM,LOSS,0,...,2018-08-11,61,3,0,AWAY_TEAM,WIN,3,NaN,NaN,NaN
4,20004,67,2018,2018-08-11,67,1,2,AWAY_TEAM,LOSS,0,...,2018-08-11,73,2,1,AWAY_TEAM,WIN,3,NaN,NaN,NaN


In [2]:
import pandas as pd
import sqlite3

# load raw matches

connect = sqlite3.connect("../db/pl_data.db")
matches = pd.read_sql_query("SELECT * FROM matches", connect)
connect.close()

matches["utc_date"] = pd.to_datetime(matches["utc_date"], dayfirst=True)

# one row per match

matches = matches.sort_values(["season_year", "utc_date"]).reset_index(drop=True)
matches["matchday"] = matches.groupby("season_year").cumcount() + 1

# odds, simple form

matches["implied_prob_home"] = 1 / matches["b365_home_odds"]
matches["implied_prob_draw"] = 1 / matches["b365_draw_odds"]
matches["implied_prob_away"] = 1 / matches["b365_away_odds"]


# BUILD HOME / AWAY TEAM-PERSPECTIVE TABLES
# each match becomes 2 rows: one for the home team's experience,
# one for the away team's experience, using shared generic column names

home = matches[[
    "match_id", "season_year", "matchday", "utc_date",
    "home_team_id", "away_team_id",
    "home_goals_fulltime", "away_goals_fulltime",
    "home_shots", "away_shots",
    "home_shots_target", "away_shots_target",
    "home_corners", "away_corners",
    "home_yellow", "away_yellow",
    "home_red", "away_red",
    "winner"
]].copy()
home.columns = [
    "match_id", "season_year", "matchday", "date",
    "team_id", "opponent_id",
    "goals_for", "goals_against",
    "shots_for", "shots_against",
    "shots_target_for", "shots_target_against",
    "corners_for", "corners_against",
    "yellow_for", "yellow_against",
    "red_for", "red_against",
    "winner"
]
home["venue"] = "HOME"

away = matches[[
    "match_id", "season_year", "matchday", "utc_date",
    "away_team_id", "home_team_id",
    "away_goals_fulltime", "home_goals_fulltime",
    "away_shots", "home_shots",
    "away_shots_target", "home_shots_target",
    "away_corners", "home_corners",
    "away_yellow", "home_yellow",
    "away_red", "home_red",
    "winner"
]].copy()
away.columns = [
    "match_id", "season_year", "matchday", "date",
    "team_id", "opponent_id",
    "goals_for", "goals_against",
    "shots_for", "shots_against",
    "shots_target_for", "shots_target_against",
    "corners_for", "corners_against",
    "yellow_for", "yellow_against",
    "red_for", "red_against",
    "winner"
]
away["venue"] = "AWAY"

# translate raw HOME_TEAM/AWAY_TEAM/DRAW into each side's own result
home_result_map = {"HOME_TEAM": "WIN", "AWAY_TEAM": "LOSS", "DRAW": "DRAW"}
away_result_map = {"HOME_TEAM": "LOSS", "AWAY_TEAM": "WIN", "DRAW": "DRAW"}

home["result"] = home["winner"].map(home_result_map)
away["result"] = away["winner"].map(away_result_map)

points_map = {"WIN": 3, "DRAW": 1, "LOSS": 0}
home["points"] = home["result"].map(points_map)
away["points"] = away["result"].map(points_map)

team_matches = pd.concat([home, away], ignore_index=True)
team_matches = team_matches.sort_values(["team_id", "date"]).reset_index(drop=True)

# rest days between games

team_matches["rest_days"] = (
    team_matches.groupby("team_id")["date"].diff().dt.days
)

## rolling form excluding current game

rolling_cols = [
    "points", "goals_for", "goals_against",
    "shots_for", "shots_against",
    "shots_target_for", "shots_target_against",
    "corners_for", "corners_against",
    "yellow_for", "yellow_against",
    "red_for", "red_against",
]

for col in rolling_cols:
    team_matches[f"rolling_{col}_5"] = (
        team_matches.groupby("team_id")[col].shift(1).rolling(window=5).sum()
    )

# HOME/AWAY specific rolling form

for col in ["points", "goals_for", "goals_against"]:
    team_matches[f"rolling_{col}_5_by_venue"] = (
        team_matches.groupby(["team_id", "venue"])[col]
        .shift(1)
        .rolling(window=5)
        .sum()
    )

# Head 2 Head form

team_matches = team_matches.sort_values(["team_id", "opponent_id", "date"])
team_matches["h2h_points_avg_3"] = (
    team_matches.groupby(["team_id", "opponent_id"])["points"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

# restore chronological order per team before moving on
team_matches = team_matches.sort_values(["team_id", "date"]).reset_index(drop=True)


# RESHAPE BACK TO ONE ROW PER MATCH
# merge the home-side and away-side feature rows back together,
# using match_id + which team was home/away in the original matches table

home_features = matches[["match_id", "home_team_id"]].merge(
    team_matches,
    left_on=["match_id", "home_team_id"],
    right_on=["match_id", "team_id"]
)

away_features = matches[["match_id", "away_team_id"]].merge(
    team_matches,
    left_on=["match_id", "away_team_id"],
    right_on=["match_id", "team_id"]
)

match_features = home_features.merge(
    away_features,
    on="match_id",
    suffixes=("_home", "_away")
)

# attach match-level fields that don't belong to either side specifically
match_level_cols = matches[[
    "match_id", "season_year", "matchday", "utc_date", "status", "referee",
    "home_goals_fulltime", "away_goals_fulltime", "winner",
    "implied_prob_home", "implied_prob_draw", "implied_prob_away"
]]

match_features = match_level_cols.merge(match_features, on="match_id")

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

team_matches[team_matches["team_id"] == 64]

In [6]:
mf = pd.read_csv("../data/processed/match_features.csv")
mf["rest_days_home"].describe()
mf["long_layoff_home"].sum()

np.int64(86)

In [19]:
mf = pd.read_csv("../data/processed/match_features.csv")
missing = mf[mf.filter(like="rolling").isna().any(axis=1)]
missing[["season_year", "home_team_id", "away_team_id"]]

,season_year,home_team_id,away_team_id
7,2018,57,65
11,2018,61,57


In [21]:
missing.filter(like="rolling").isna().sum(axis=1)

7     16
11     3
dtype: int64